In [1]:
import sys
import os
import math
import numpy as np

states = { "s": 0, "E": 1, "5": 2, "I" : 3, "e": 4}
id2state = {0: "s", 1: "E", 2: "5", 3: "I", 4: "e"}

state_transition_prob = np.array([[0.0, 1.0, 0.0, 0.0, 0.0], 
                                  [0.0, 0.9, 0.1, 0.0, 0.0], 
                                  [0.0, 0.0, 0.0, 1.0, 0.0],
                                  [0.0, 0.0, 0.0, 0.9, 0.1],
                                  [0.0, 0.0, 0.0, 0.0, 0.0]]) 
emission_nuc_codes = {'A': 0, 
                      'C': 1, 
                      'G': 2, 
                      'T': 3}

emission_probs = np.array([[0.00, 0.00, 0.00, 0.00], 
                           [0.25, 0.25, 0.25, 0.25],
                           [0.05, 0.00, 0.95, 0.00],
                           [0.40, 0.10, 0.10, 0.40],
                           [0.00, 0.00, 0.00, 0.00]]) 

query_sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"


In [2]:
def get_log_prob_for_state_path (state_path, query_sequence):
    res = math.log(0.25)
    for i in range(1, len(state_path)):
        res += math.log(state_transition_prob[ states[state_path[i-1]] ][ states[state_path[i]] ]*emission_probs[ states[state_path[i]] ][ emission_nuc_codes[query_sequence[i]] ])
    return res

In [3]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEE5IIIIIIIIIIIIIIIIIII
k1 = get_log_prob_for_state_path("EEEEEE5IIIIIIIIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") +  math.log (0.1)
print (k1)


-43.89740030179307


In [4]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEE5IIIIIIIIIIIIIIIII
k2 = get_log_prob_for_state_path("EEEEEEEE5IIIIIIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k2)


-43.45111319916465


In [5]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEE5IIIIIIIIIIIII
k3 = get_log_prob_for_state_path("EEEEEEEEEEEE5IIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k3)


-43.944833355027704


In [6]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEEEEE5IIIIIIIIII
k4 = get_log_prob_for_state_path("EEEEEEEEEEEEEEE5IIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k4)


-42.58225552052512


In [7]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEEEEEEEE5IIIIIII
k5 = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEE5IIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k5)


-41.21967768602254


In [8]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEEEEEEEEEEEE5III
k6 = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEEEEEE5III", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k6)


-41.713397841885595


In [9]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEEEEEEEEEEEEEEEE
only_E = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEEEEEEEEEE", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (only_E)


-40.98025137355685


### Design of the Viterbi Value matrix

Rows correspond to the hidden states, and the columns correspond to the emissions that is the observed nucleotide sequences. Here I am showing the calculation for the first two nucletides. 

```
             C                                                          T     T
s [s-s-C(0.00) max(s-s-C-s-T, s-E-C-s-T, s-5-C-s-T, s-I-C-s-T, s-e-C-s-T)     .] 
E [s-E-C(0.25) max(s-s-C-E-T, s-E-C-E-T, s-5-C-E-T, s-I-C-E-T, s-e-C-E-T)     .] 
5 [s-5-C(0.00) max(s-s-C-5-T, s-E-C-5-T, s-5-C-5-T, s-I-C-5-T, s-e-C-5-T)     .]
I [s-I-C(0.00) max(s-s-C-I-T, s-E-C-I-T, s-5-C-I-T, s-I-C-I-T  s-e-C-I-T)     .]
e [s-e-C(0.00) max(s-s-C-e-T, s-E-C-e-T, s-5-C-e-T, s-I-C-e-T, s-e-C-e-T)     .]

```

It is important to remember that you will be working in the log scale.

In [10]:
# Initiate two matrices: 
# viterbi_value_matrix: stores log probabilities
# viterbi_trace_matrix: stores traceback indices

num_states = len(states)
seq_len = len(query_sequence)

viterbi_value_matrix = np.full((num_states, seq_len), -np.inf)
viterbi_trace_matrix = np.zeros((num_states, seq_len), dtype=int)

# Initialization for first nucleotide
first_nuc = query_sequence[0]

for state_idx in range(num_states):
    transition_prob = state_transition_prob[states["s"]][state_idx]
    emission_prob = emission_probs[state_idx][emission_nuc_codes[first_nuc]]

    if transition_prob > 0 and emission_prob > 0:
        viterbi_value_matrix[state_idx][0] = (
            math.log(0.25) +
            math.log(transition_prob) +
            math.log(emission_prob)
        )

    viterbi_trace_matrix[state_idx][0] = state_idx

print(viterbi_value_matrix)
print(viterbi_trace_matrix)

[[       -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf]
 [-2.77258872        -inf        -inf        -inf        -inf        -inf
         -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf]
 [       -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf]
 [       -inf        -inf      

### Implementation of Viterbi Algorithm
Write a function `calculate_prob_for_a_node()` that populate a single cell in the matrix. The function will return two values:
1. the maximum value, for example, look at the 2nd row, 2nd column in the matrix: `max(s-s-C-E-T, s-E-C-E-T, s-5-C-E-T, s-I-C-E-T, s-e-C-E-T)`. If the probability for `s-E-C-E-T` is highest (lets say X), then the function should return `X`

**AND** 

2. The index of that maximum value described in the first point: so index of X is `1` (recall that Python works on the 0-based index system)

- Populate `viterbi_value_matrix` with `X` for row 2 and col 2

- Populate `viterbi_trace_matrix` with `1` for row 2 and col 2

In [11]:
def calculate_prob_for_a_node(row, col):
    current_nuc = query_sequence[col]

    max_prob = -np.inf
    max_state = 0

    for prev_state in range(len(states)):
        prev_prob = viterbi_value_matrix[prev_state][col - 1]

        transition_prob = state_transition_prob[prev_state][row]
        emission_prob = emission_probs[row][emission_nuc_codes[current_nuc]]

        if (
            prev_prob == -np.inf or
            transition_prob == 0 or
            emission_prob == 0
        ):
            continue

        current_prob = (
            prev_prob +
            math.log(transition_prob) +
            math.log(emission_prob)
        )

        if current_prob > max_prob:
            max_prob = current_prob
            max_state = prev_state

    return max_prob, max_state

In [12]:
# Fill the Viterbi matrices

for col in range(1, seq_len):
    for row in range(num_states):
        best_prob, best_prev_state = calculate_prob_for_a_node(row, col)

        viterbi_value_matrix[row][col] = best_prob
        viterbi_trace_matrix[row][col] = best_prev_state

print(viterbi_value_matrix)
print(viterbi_trace_matrix)

[[        -inf         -inf         -inf         -inf         -inf
          -inf         -inf         -inf         -inf         -inf
          -inf         -inf         -inf         -inf         -inf
          -inf         -inf         -inf         -inf         -inf
          -inf         -inf         -inf         -inf         -inf
          -inf]
 [ -2.77258872  -4.2642436   -5.75589848  -7.24755335  -8.73920823
  -10.23086311 -11.72251798 -13.21417286 -14.70582774 -16.19748261
  -17.68913749 -19.18079237 -20.67244724 -22.16410212 -23.655757
  -25.14741187 -26.63906675 -28.13072163 -29.6223765  -31.11403138
  -32.60568626 -34.09734113 -35.58899601 -37.08065089 -38.57230576
  -40.06396064]
 [        -inf         -inf         -inf         -inf -12.54587072
          -inf -12.58474149         -inf -15.56805125 -20.0041451
  -21.49579998 -22.98745486 -21.53467075         -inf -27.46241949
  -26.00963538 -30.44572924         -inf -30.48460001         -inf
  -36.41234875 -37.90400362 -36.4

In [13]:
# Traceback function

def traceback_state_path():
    last_col = seq_len - 1

    final_state = np.argmax(viterbi_value_matrix[:, last_col])

    path = [final_state]

    current_state = final_state

    for col in range(last_col, 0, -1):
        current_state = viterbi_trace_matrix[current_state][col]
        path.append(current_state)

    path.reverse()

    state_path = "".join([id2state[state] for state in path])

    return state_path


best_path = traceback_state_path()

print("Most probable state path:")
print(best_path)

Most probable state path:
EEEEEEEEEEEEEEEEEEEEEEEEEE
